# LabSD E1 — Cascade Degradation

Thin orchestrator. Heavy logic lives in the `labsd` package (Kaggle Dataset `ifty1011/labsd-src`).

## Phases
1. Env check + install
1.5. Extract nuScenes mini tarball to writable scratch
2. Build Boston/Singapore splits (sanity check on real data)
3. Train C1 (CenterPoint) on Boston
4. Train C2 (MTP) on Boston
5. Set up C3 (IDM planner)
6. Baseline 5-measurement run on Singapore
7. Retrain C1 on Singapore
8. Post-retrain 5-measurement run
9. Table I + cascade diagnostic + plot

## Required attached datasets
- `ifty1011/nuscenes-mini` — 4 GB tarball at `/kaggle/input/nuscenes-mini/v1.0-mini.tgz`
- `ifty1011/labsd-src` — our Python package, pip-installable

## Smoke-test mode
Set `SMOKE_TEST = True` (cell below) to run only Phases 1–2 and exit. Used for the first end-to-end validation before kicking off training.

In [ ]:
SMOKE_TEST = False   # we've validated the platform; full run now


## Phase 1 — env check

In [ ]:
import os, sys, torch
print('python:', sys.version.split()[0])
print('torch :', torch.__version__)
print('cuda  :', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
print('---')
print('contents of /kaggle/input/:')
for entry in sorted(os.listdir('/kaggle/input')):
    sub = os.path.join('/kaggle/input', entry)
    inner = sorted(os.listdir(sub))[:8] if os.path.isdir(sub) else '(file)'
    print(f'  {entry}/ → {inner}')
print('---')
!df -h /kaggle/working

In [ ]:
# Phase 1 — make labsd importable, robust to dataset packing AND mount layout
import os, sys, tarfile, zipfile, shutil, glob, subprocess
from pathlib import Path

LABSD_DIR = '/kaggle/working/labsd_pkg'
Path(LABSD_DIR).mkdir(parents=True, exist_ok=True)

def _locate_labsd_pkg():
    """Return a directory containing an importable `labsd` package.

    Robust to every packing/mount layout we have hit: a plain `labsd/` dir, a
    `labsd.tar`/`labsd.zip` archive, and Kaggle's auto-extraction which can add
    an extra nesting level (labsd/labsd/__init__.py).
    """
    # 1) any already-extracted labsd/__init__.py anywhere under /kaggle/input
    for init in glob.glob('/kaggle/input/**/labsd/__init__.py', recursive=True):
        parent = str(Path(init).parent.parent)   # dir that CONTAINS labsd/
        print('found extracted package via', init)
        return parent
    # 2) an archive we can unpack ourselves
    for tar in glob.glob('/kaggle/input/**/labsd.tar', recursive=True):
        print('extracting', tar)
        with tarfile.open(tar) as tf:
            tf.extractall(LABSD_DIR)
        for init in glob.glob(LABSD_DIR + '/**/labsd/__init__.py', recursive=True):
            return str(Path(init).parent.parent)
    for zf_path in glob.glob('/kaggle/input/**/labsd.zip', recursive=True):
        print('extracting', zf_path)
        with zipfile.ZipFile(zf_path) as zf:
            zf.extractall(LABSD_DIR)
        for init in glob.glob(LABSD_DIR + '/**/labsd/__init__.py', recursive=True):
            return str(Path(init).parent.parent)
    return None

PKG_PARENT = _locate_labsd_pkg()
if PKG_PARENT is None:
    raise RuntimeError(
        'labsd package not found. /kaggle/input tree:\n'
        + os.popen('find /kaggle/input -maxdepth 4 -name "*.py" -o -maxdepth 4 -type d').read()[:3000]
    )
if PKG_PARENT not in sys.path:
    sys.path.insert(0, PKG_PARENT)
import labsd
print('labsd loaded:', labsd.__file__)
from labsd import class_balance as _cb   # fail fast if the mirror is stale
print('class_balance present OK')

# nuscenes-devkit: install with --no-deps to avoid downgrading Kaggle's
# numpy/scipy/sklearn. Then pull in the few extra deps it actually needs
# (mostly small pure-python packages that don't conflict).
print('\ninstalling nuscenes-devkit (no-deps) ...')
ret = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--no-deps', 'nuscenes-devkit'],
    capture_output=True, text=True,
)
if ret.returncode != 0:
    print('STDOUT:', ret.stdout[-1500:]); print('STDERR:', ret.stderr[-1500:])
    raise RuntimeError('nuscenes-devkit install failed')
print(ret.stdout.strip().split('\n')[-1])

# Auxiliary deps not bundled in Kaggle's image. Pin pyquaternion which is tiny.
ret = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
     'pyquaternion', 'cachetools', 'descartes'],
    capture_output=True, text=True,
)
if ret.returncode != 0:
    print('STDERR:', ret.stderr[-1500:])
    raise RuntimeError('aux deps install failed')
print('aux deps OK')

# Smoke import — fail fast if anything is still missing
from nuscenes.nuscenes import NuScenes
import nuscenes
print('nuscenes-devkit version:', getattr(nuscenes, '__version__', 'unknown'))

# SOTA — install planning-centric-metrics (Philion et al., CVPR 2020)
ret = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--no-deps',
     'planning-centric-metrics'],
    capture_output=True, text=True,
)
if ret.returncode != 0:
    print('PKL install warning (continuing without PKL):', ret.stderr[-500:])
    PKL_AVAILABLE = False
else:
    try:
        import planning_centric_metrics
        PKL_AVAILABLE = True
        print('PKL package OK')
    except Exception as e:
        print('PKL import failed:', e)
        PKL_AVAILABLE = False

# Real C1 — install Ultralytics for YOLOv11.
ret = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--no-deps',
     'ultralytics'],
    capture_output=True, text=True,
)
if ret.returncode == 0:
    # Ultralytics has its own deps that may not be present.
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    '--no-cache-dir', 'ultralytics-thop', 'py-cpuinfo'],
                   capture_output=True, text=True)
    try:
        from ultralytics import YOLO
        YOLO_AVAILABLE = True
        print('ultralytics OK')
    except Exception as e:
        print('ultralytics import failed:', e)
        YOLO_AVAILABLE = False
else:
    print('ultralytics install failed:', ret.stderr[-500:])
    YOLO_AVAILABLE = False

## Phase 1.5 — extract nuScenes mini tarball

`/kaggle/input/` is read-only and contains the compressed `v1.0-mini.tgz`. We extract it once to `/kaggle/working/nuscenes/` (persistent across sessions when notebook persistence is enabled). Subsequent runs skip the extract.

In [ ]:
import os, tarfile, time, glob
from pathlib import Path

candidates = [
    '/kaggle/input/nuscenes-mini/v1.0-mini.tgz',
    '/kaggle/input/datasets/ifty1011/nuscenes-mini/v1.0-mini.tgz',
]
candidates += glob.glob('/kaggle/input/**/v1.0-mini.tgz', recursive=True)
TARBALL = next((p for p in candidates if os.path.exists(p)), None)
assert TARBALL, f'tarball missing. /kaggle/input/ tree:\n' + os.popen('find /kaggle/input -maxdepth 4').read()
print('found TARBALL =', TARBALL)

NUSC_ROOT = '/kaggle/working/nuscenes'
MARKER = Path(NUSC_ROOT) / '.extracted'
Path(NUSC_ROOT).mkdir(parents=True, exist_ok=True)

if MARKER.exists():
    print('already extracted, skipping.')
else:
    print(f'extracting {TARBALL} → {NUSC_ROOT} ...')
    t0 = time.time()
    with tarfile.open(TARBALL, 'r:gz') as tf:
        tf.extractall(NUSC_ROOT)
    MARKER.touch()
    print(f'done in {time.time()-t0:.1f}s')

!ls $NUSC_ROOT
print('---')
!ls $NUSC_ROOT/v1.0-mini/ 2>/dev/null | head -10

## Phase 2 — build Boston/Singapore splits

Sanity check on real metadata: confirms nuscenes-devkit can load the dataset and the location-based partition produces the expected ~6 Boston / ~4 Singapore.

In [ ]:
from labsd.splits import build_splits
from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap

SPLITS_JSON = '/kaggle/working/splits/splits.json'
splits = build_splits(
    nusc_root=NUSC_ROOT,
    out_path=SPLITS_JSON,
)
for k, v in splits.items():
    print(f'{k:18s}: {len(v):2d} scene(s)')
assert sum(len(v) for v in splits.values()) == 10

NUSC = NuScenes(version='v1.0-mini', dataroot=NUSC_ROOT, verbose=False)
print('NuScenes loaded:', len(NUSC.scene), 'scenes')

# Load all map locations referenced by mini scenes (PKL needs them).
import os
MAP_LOCATIONS = ['boston-seaport', 'singapore-onenorth',
                 'singapore-hollandvillage', 'singapore-queenstown']
NUSC_MAPS = {}
for loc in MAP_LOCATIONS:
    try:
        NUSC_MAPS[loc] = NuScenesMap(dataroot=NUSC_ROOT, map_name=loc)
    except Exception as e:
        print(f'map {loc}: {type(e).__name__}: {e}')
print('maps loaded:', list(NUSC_MAPS.keys()))

print('splits OK')

In [ ]:
# Smoke-test gate — stop here if SMOKE_TEST is on
if SMOKE_TEST:
    raise SystemExit('SMOKE_TEST: env + extract + splits all OK. Set SMOKE_TEST=False to continue.')

## Phase 3 — train C1 (CenterPoint) on Boston

In [ ]:
# Phase 3 — REAL C1 — YOLOv11 fine-tuned on Boston camera images
from labsd.c1_yolo import (
    build_yolo_dataset, write_yolo_data_yaml,
    fine_tune_yolo, c1_descriptor_yolo,
)
from pathlib import Path
import os, urllib.request

BOSTON_DATA_DIR = '/kaggle/working/yolo_boston'
n_train = build_yolo_dataset(NUSC, splits['boston_train'],
                             out_dir=BOSTON_DATA_DIR, split_name='train')
n_val = build_yolo_dataset(NUSC, splits['boston_val'],
                           out_dir=BOSTON_DATA_DIR, split_name='val')
print(f'Boston: {n_train} train images, {n_val} val images')
data_yaml = write_yolo_data_yaml(BOSTON_DATA_DIR)

# Pre-fetch yolo11n.pt to /kaggle/working so ultralytics doesn't
# attempt a download into a read-only path.
WEIGHTS = '/kaggle/working/yolo11n.pt'
if not os.path.exists(WEIGHTS):
    urllib.request.urlretrieve(
        'https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt',
        WEIGHTS,
    )

if YOLO_AVAILABLE and n_train > 0 and n_val > 0:
    boston_weights = fine_tune_yolo(
        data_yaml=data_yaml,
        out_dir='/kaggle/working/c1_boston_yolo',
        base_weights=WEIGHTS,
        epochs=10, imgsz=640, batch=8, device='0',
        name='c1_boston', seed=0,   # pinned for a stable baseline across runs
    )
    c1_boston = c1_descriptor_yolo(
        '/kaggle/working/c1_boston', boston_weights, label='boston_yolo',
        val_data_yaml=data_yaml)
    print('C1 Boston (YOLO) descriptor:', c1_boston)
else:
    print('YOLO unavailable or no data — falling back to perturbation oracle')
    from labsd.train_c1 import train_c1
    from labsd.c1_perturbation import PERTURB_NONE
    c1_boston = train_c1(work_dir='/kaggle/working/c1_boston', profile=PERTURB_NONE)

## Phase 4 — train C2 (MTP) on Boston

In [ ]:
# Phase 4 — C2 = constant-velocity model descriptor
from labsd.train_c2 import train_c2
c2_boston = train_c2(split='boston_train', out_path='/kaggle/working/c2_boston.json')
print('C2 descriptor:', c2_boston)

## Phase 6 — baseline 5-measurement run on Singapore

In [ ]:
# Phase 6 — baseline 5-measurement run on Singapore (Boston-trained C1)
from labsd.eval import run_all_measurements
baseline = run_all_measurements(
    c1_ckpt=c1_boston, c2_ckpt=c2_boston,
    split='singapore_val',
    out_path='/kaggle/working/baseline.json',
    nusc=NUSC, splits_json=SPLITS_JSON,
    nusc_maps=NUSC_MAPS, enable_pkl=PKL_AVAILABLE,
)
import json; print(json.dumps(baseline, indent=2))

## Phase 7 — retrain C1 on Singapore

In [ ]:
# Phase 7 — REAL C1 retrain on Singapore (fine-tune Boston weights)
from labsd.c1_yolo import (
    build_yolo_dataset, write_yolo_data_yaml,
    fine_tune_yolo, c1_descriptor_yolo,
)
import json as _json

SG_DATA_DIR = '/kaggle/working/yolo_singapore'
n_train_sg = build_yolo_dataset(NUSC, splits['singapore_train'],
                                out_dir=SG_DATA_DIR, split_name='train')
n_val_sg = build_yolo_dataset(NUSC, splits['singapore_val'],
                              out_dir=SG_DATA_DIR, split_name='val')
print(f'Singapore: {n_train_sg} train images, {n_val_sg} val images')
data_yaml_sg = write_yolo_data_yaml(SG_DATA_DIR)

_descr = _json.load(open(c1_boston))
if YOLO_AVAILABLE and _descr.get('kind') == 'yolo' and n_train_sg > 0:
    sg_weights = fine_tune_yolo(
        data_yaml=data_yaml_sg,
        out_dir='/kaggle/working/c1_sg_yolo',
        base_weights=_descr['weights_path'],   # start from Boston weights
        epochs=10, imgsz=640, batch=8, device='0',
        name='c1_singapore',
    )
    c1_singapore = c1_descriptor_yolo(
        '/kaggle/working/c1_singapore', sg_weights, label='singapore_yolo',
        val_data_yaml=data_yaml_sg)
    print('C1 Singapore (YOLO fine-tuned) descriptor:', c1_singapore)
else:
    print('Falling back to perturbation knob for C1 retrain')
    from labsd.train_c1 import train_c1
    from labsd.c1_perturbation import PERTURB_SG_FT
    c1_singapore = train_c1(
        work_dir='/kaggle/working/c1_singapore',
        profile=PERTURB_SG_FT,
        resume_from=c1_boston,
    )

## Phase 8 — post-retrain 5-measurement run

In [ ]:
# Phase 8 — post-retrain 5-measurement run
after = run_all_measurements(
    c1_ckpt=c1_singapore, c2_ckpt=c2_boston,
    split='singapore_val',
    out_path='/kaggle/working/after_retrain.json',
    nusc=NUSC, splits_json=SPLITS_JSON,
    nusc_maps=NUSC_MAPS, enable_pkl=PKL_AVAILABLE,
)
import json; print(json.dumps(after, indent=2))

## Phase 9 — Table I + diagnostic + plot

In [ ]:
from labsd.table import build_table, cascade_diagnostic
df = build_table(
    baseline_json='/kaggle/working/baseline.json',
    after_json='/kaggle/working/after_retrain.json',
    out_csv='/kaggle/working/table_I.csv',
)
print(df.to_string(index=False))
print()
print('diagnostic:', cascade_diagnostic(df))

In [ ]:
# Phase 9 — full presentation figure set (Tier 1 + Tier 2)
from labsd.figures import generate_all
import json

# YOLO run dirs (where Ultralytics wrote results.png + confusion_matrix.png)
yolo_boston_dir = '/kaggle/working/c1_boston_yolo/c1_boston'
yolo_sg_dir     = '/kaggle/working/c1_sg_yolo/c1_singapore'

boston_descr = json.load(open(c1_boston))
sg_descr     = json.load(open(c1_singapore))

fig_paths = generate_all(
    baseline_json='/kaggle/working/baseline.json',
    after_json='/kaggle/working/after_retrain.json',
    yolo_boston_dir=yolo_boston_dir,
    yolo_singapore_dir=yolo_sg_dir,
    boston_weights=boston_descr['weights_path'],
    singapore_weights=sg_descr['weights_path'],
    nusc=NUSC,
    splits_json=SPLITS_JSON,
    out_dir='/kaggle/working/figs',
)

print(f'\nGenerated {len(fig_paths)} figures:')
for p in fig_paths:
    print(f'  {p}')

# Inline-display the most important ones
from IPython.display import Image, display
for fname in ['fig03_pipeline_diagram.png',
              'fig01_table_full.png',
              'fig02_cascade_planning.png',
              'fig08_detection_grid.png',
              'fig09_per_scene_l2.png',
              'fig10_failure_modes.png']:
    p = f'/kaggle/working/figs/{fname}'
    import os
    if os.path.exists(p):
        display(Image(p))

## Phase 10 — Q1 campaign (multiple fine-tuning scenarios)
Sensei Q1: do different fine-tuning campaigns produce different entanglements?
Loops the harness over dataset-size x epochs x seed and labels each regime.

In [ ]:
# Phase 10 — Q1 campaign: multiple fine-tuning scenarios
from labsd.campaign import run_campaign, default_matrix
from labsd.c1_yolo import build_yolo_dataset, write_yolo_data_yaml
import json as _json

# Full Singapore YOLO dataset (reuse Phase-7 build if present, else build now).
SG_FULL_DIR = '/kaggle/working/yolo_singapore'
try:
    _ = SG_DATA_DIR  # built in Phase 7
    SG_FULL_DIR = SG_DATA_DIR
except NameError:
    n_tr = build_yolo_dataset(NUSC, splits['singapore_train'],
                              out_dir=SG_FULL_DIR, split_name='train')
    n_va = build_yolo_dataset(NUSC, splits['singapore_val'],
                              out_dir=SG_FULL_DIR, split_name='val')
    write_yolo_data_yaml(SG_FULL_DIR)
    print(f'built full SG dataset: {n_tr} train, {n_va} val')

cfgs = default_matrix()
print(f'campaign: {len(cfgs)} configs')

result = run_campaign(
    nusc=NUSC, splits=splits, splits_json=SPLITS_JSON,
    c1_boston_descriptor=c1_boston,      # frozen "before" C1
    c2_ckpt=c2_boston,                   # frozen C2
    baseline_json='/kaggle/working/baseline.json',
    sg_full_dataset_dir=SG_FULL_DIR,
    work_root='/kaggle/working/campaign',
    configs=cfgs,
    nusc_maps=NUSC_MAPS, enable_pkl=False,   # PKL off for speed across many runs
)

# Compact summary table
print(f"{'tag':<16}{'n_tr':>5}{'d1':>9}{'D2':>9}{'D3':>9}{'pshift':>8}  regime")
for r in result['rows']:
    ps = r.get('plan_shift3')
    ps_s = f"{ps:.3f}" if ps is not None else "  n/a"
    print(f"{r['tag']:<16}{r['n_train']:>5}"
          f"{(r['delta1'] or 0):>+9.4f}{(r['Delta2'] or 0):>+9.3f}"
          f"{(r['Delta3'] or 0):>+9.3f}{ps_s:>8}  {r['regime']}")

# Regime counts
from collections import Counter
print('regime counts:', dict(Counter(r['regime'] for r in result['rows'])))
_json.dump(result, open('/kaggle/working/campaign_results.json','w'), indent=2)
print('wrote /kaggle/working/campaign_results.json')


## Phase 11 — Q2: drift-gate retrospective evaluation
Does the C1->C2 interface drift gate flag the campaigns that actually moved the plan most?

In [ ]:
# Phase 11 — Q2: evaluate the drift gate on the finished campaign
import traceback, json as _json
try:
    from labsd.campaign import evaluate_gate_on_campaign

    _bd = _json.load(open(c1_boston))
    boston_weights = _bd['weights_path']

    gate_eval = evaluate_gate_on_campaign(
        nusc=NUSC, splits=splits,
        campaign_result=result,
        c1_boston_weights=boston_weights,
        work_root='/kaggle/working/campaign',
        tau=0.25,
    )

    print(f"Spearman(drift, plan_shift) = {gate_eval['spearman_drift_vs_planshift']}")
    print(f"{'config':<16}{'drift':>8}{'risk':>6}{'plan_shift':>12}{'D3':>8}")
    for r in sorted(gate_eval['rows'], key=lambda x: -(x['drift'] or 0)):
        ps = r['plan_shift3']; ps_s = f'{ps:.3f}' if ps is not None else 'n/a'
        print(f"{r['tag']:<16}{r['drift']:>8.3f}{('YES' if r['cascade_risk'] else 'no'):>6}{ps_s:>12}{(r['Delta3'] or 0):>+8.3f}")

    _json.dump(gate_eval, open('/kaggle/working/gate_eval.json','w'), indent=2)
    print('wrote gate_eval.json')
except Exception as e:
    tb = traceback.format_exc()
    print('PHASE 11 FAILED:')
    print(tb)
    open('/kaggle/working/gate_error.txt','w').write(tb)


## Phase 12 — Class-balanced fine-tune (chasing $\delta_1 > 0$)

Every campaign run lost aggregate mAP because the 119-image Singapore set is
dominated by `car`. Here we oversample rare-class images so the class mix
flattens, aiming for a run where C1 **improves** on its own metric while the
system still degrades — the strict entangled-enhancement condition.

In [ ]:
# Phase 12 — class-balanced fine-tune: does delta1 turn positive?
import traceback, json as _json
try:
    from labsd.campaign import run_class_balanced
    from labsd.class_balance import class_counts

    print('Singapore train class mix (class_idx: instances):')
    print(dict(sorted(class_counts(SG_FULL_DIR, 'train').items())))

    cb = run_class_balanced(
        nusc=NUSC, splits=splits, splits_json=SPLITS_JSON,
        c1_boston_descriptor=c1_boston,
        c2_ckpt=c2_boston,
        baseline_json='/kaggle/working/baseline.json',
        sg_full_dataset_dir=SG_FULL_DIR,
        work_root='/kaggle/working/balanced',
        epochs_list=(10, 20), seeds=(1, 2, 3), powers=(0.5, 1.0),
        nusc_maps=NUSC_MAPS, enable_pkl=False,
    )

    print()
    print(f"{'tag':<22}{'d1':>9}{'D2':>9}{'D3':>9}{'pshift':>8}  strictEE")
    for r in cb['rows']:
        ps = r.get('plan_shift3'); ps_s = f'{ps:.2f}' if ps is not None else ' n/a'
        print(f"{r['tag']:<22}{(r['delta1'] or 0):>+9.4f}{(r['Delta2'] or 0):>+9.3f}"
              f"{(r['Delta3'] or 0):>+9.3f}{ps_s:>8}  {r['strict_entangled_enhancement']}")
    print()
    print('delta1>0 runs :', cb['n_delta1_positive'], '/', len(cb['rows']))
    print('STRICT EE runs:', cb['n_strict_entangled_enhancement'], '/', len(cb['rows']))
    _json.dump(cb, open('/kaggle/working/class_balanced_results.json','w'), indent=2)
    print('wrote class_balanced_results.json')
except Exception as e:
    tb = traceback.format_exc()
    print('PHASE 12 FAILED:')
    print(tb)
    open('/kaggle/working/phase12_error.txt','w').write(tb)
